# 00 - Process Single Year Lung Cancer Mortality Data

## Motivation

In this project, we are predicting county-level lung cancer mortality rates in the United States.

We use **age-standardized mortality rates** to ensure fair comparisons across counties with different age distributions.

This notebook extracts the lung cancer mortality data for each individual year (2012-2019), which will later be combined into a single dataset.


In [1]:
import pandas as pd
import os
import glob


## Processing Function

The function below extracts lung cancer mortality data for a single year:
- Filters for `metric_name == 'Rate'`
- Filters for `sex_name == 'Both'` (combined sexes)
- Filters for `race_name == 'Total'` (aggregate across all races)
- Filters for `age_name == 'Age-standardized'` (age-standardized mortality rate)
- Keeps only county-level data (`fips > 60`)
- Removes unnecessary columns and renames the target variable


In [2]:
def process_lung_cancer_mortality_data(year):
    """
    Processes lung cancer mortality data for a specific year.

    Parameters:
    year (int): The year for which the data needs to be processed (e.g., 2012).

    Returns:
    pd.DataFrame: Processed lung cancer mortality data for the specified year.
    """
    input_pattern = f"../data/raw/IHME_USA_LUNG_CANCER_BOTH/IHME_USA_LUNG_CANCER_COUNTY_RACE_ETHNICITY_2000_2019_MX_{year}_BOTH_*.CSV"
    matching_files = sorted(glob.glob(input_pattern))

    if len(matching_files) != 1:
        raise FileNotFoundError(f"Expected 1 file for year {year}, found {len(matching_files)}: {matching_files}")

    input_file_path = matching_files[0]

    df = pd.read_csv(input_file_path)
    print(f"Loaded {len(df)} rows for year {year}")
    print(f"Matched raw file: {os.path.basename(input_file_path)}")

    df_filtered = df[
        (df['metric_name'] == 'Rate') &
        (df['sex_name'] == 'Both') &
        (df['race_name'] == 'Total') &
        (df['age_name'] == 'Age-standardized')
    ]
    print(f"After filtering for Rate, Both, Total race, and Age-standardized: {len(df_filtered)} rows")

    df_clean = df_filtered.dropna().copy()
    print(f"After dropping NaN: {len(df_clean)} rows")

    df_clean['fips'] = pd.to_numeric(df_clean['fips'], errors='coerce')
    df_counties = df_clean[df_clean['fips'] > 60].copy()
    print(f"After filtering for county-level (fips > 60): {len(df_counties)} rows")

    columns_to_drop = ['measure_id', 'location_id', 'fips', 'measure_name', 'race_id',
                       'race_name', 'sex_id', 'sex_name', 'age_group_id', 'age_name',
                       'cause_id', 'cause_name', 'metric_id', 'metric_name', 'upper', 'lower']
    df_final = df_counties.drop(columns=columns_to_drop)

    df_final = df_final.rename(columns={'val': 'lung_cancer_mortality_rate'})

    output_file_path = f"../data/processed/lung_cancer_single_year/lung_cancer_mortality_{year}.csv"
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    df_final.to_csv(output_file_path, index=False)

    print(f"Processed lung cancer mortality data for {year} saved to {output_file_path}")
    print(f"Final dataset shape: {df_final.shape}")
    print("---")

    return df_final

## Test with a Single Year


In [3]:
# Test with 2012 first
df_2012 = process_lung_cancer_mortality_data(2012)
df_2012.head()

Loaded 404460 rows for year 2012
Matched raw file: IHME_USA_LUNG_CANCER_COUNTY_RACE_ETHNICITY_2000_2019_MX_2012_BOTH_Y2025M06D15.CSV
After filtering for Rate, Both, Total race, and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed lung cancer mortality data for 2012 saved to ../data/processed/lung_cancer_single_year/lung_cancer_mortality_2012.csv
Final dataset shape: (3127, 3)
---


,location_name,year,lung_cancer_mortality_rate
8292,Autauga County (Alabama),2012,0.000756
8298,Baldwin County (Alabama),2012,0.000620
8304,Barbour County (Alabama),2012,0.000790
8310,Bibb County (Alabama),2012,0.000827
8316,Blount County (Alabama),2012,0.000754


In [4]:
# Check statistics of the lung cancer mortality rate
print("Lung Cancer Mortality Rate Statistics (2012):")
print(df_2012['lung_cancer_mortality_rate'].describe())

Lung Cancer Mortality Rate Statistics (2012):
count    3127.000000
mean        0.000624
std         0.000157
min         0.000134
25%         0.000507
50%         0.000620
75%         0.000725
max         0.001501
Name: lung_cancer_mortality_rate, dtype: float64


## Process All Years (2012-2019)

We process years 2012-2019 to match the time period of our ACS and atmospheric data.


In [5]:
# Process all years from 2012 to 2019
for year in range(2012, 2020):
    process_lung_cancer_mortality_data(year)

Loaded 404460 rows for year 2012
Matched raw file: IHME_USA_LUNG_CANCER_COUNTY_RACE_ETHNICITY_2000_2019_MX_2012_BOTH_Y2025M06D15.CSV
After filtering for Rate, Both, Total race, and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed lung cancer mortality data for 2012 saved to ../data/processed/lung_cancer_single_year/lung_cancer_mortality_2012.csv
Final dataset shape: (3127, 3)
---
Loaded 404460 rows for year 2013
Matched raw file: IHME_USA_LUNG_CANCER_COUNTY_RACE_ETHNICITY_2000_2019_MX_2013_BOTH_Y2025M06D15.CSV
After filtering for Rate, Both, Total race, and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed lung cancer mortality data for 2013 saved to ../data/processed/lung_cancer_single_year/lung_cancer_mortality_2013.csv
Final dataset shape: (3127, 3)
---
Loaded 404460 rows for year 2014
Matched raw file: IHME_USA_LUNG_CANCER_COUNTY_RACE_

## Verify Output Files


In [6]:
# List all generated files
output_dir = "../data/processed/lung_cancer_single_year/"
files = sorted(os.listdir(output_dir))
print("Generated files:")
for f in files:
    if f.endswith('.csv'):
        filepath = os.path.join(output_dir, f)
        df = pd.read_csv(filepath)
        print(f"  {f}: {len(df)} rows")


Generated files:
  lung_cancer_mortality_2012.csv: 3127 rows
  lung_cancer_mortality_2013.csv: 3127 rows
  lung_cancer_mortality_2014.csv: 3127 rows
  lung_cancer_mortality_2015.csv: 3127 rows
  lung_cancer_mortality_2016.csv: 3127 rows
  lung_cancer_mortality_2017.csv: 3127 rows
  lung_cancer_mortality_2018.csv: 3127 rows
  lung_cancer_mortality_2019.csv: 3127 rows
